# 06 unseen subtypes, distributional abstention gate, onset localisation

Design decisions:
- Unseen-subtype evaluation (E2): for each of the eight subtypes, one pipeline is fitted per seed on a single training set and calibrated on a single calibration set; the seen flights of the affected family are split into disjoint train, calibration and seen-test thirds; other families keep 20 train and 20 calibration flights. The same fitted pipeline and the same conformal thresholds are evaluated on the seen-test flights and on the withheld subtype, so model, training volume and thresholds are held fixed and only familiarity of the test subtype differs. Three seeds; mean and standard deviation reported.
- Distributional abstention gate (E4): Mahalanobis distance with Ledoit-Wolf covariance and an isolation forest on the standardised flight aggregates, fitted on the training flights, thresholds at the 95th percentile of the calibration flights; withheld fraction reported on seen-test and withheld flights, alone and combined with the conformal policy; threshold-free separability (AUROC of each score for withheld against seen-test flights, and the catch rate at a 5 percent false-abstention budget), with one minus confidence and a declared gap-rule (any availability aggregate above its calibration 95th percentile) as baselines; raw scores exported to `e2_gate_scores.csv`.
- Onset localisation (E5): one pipeline (20 train / 20 calibration per family), the declared alert rule and a CUSUM baseline (k = 0.5, h = 0.75) (window probability of any non-nominal class above 0.5 in two consecutive windows) on the remaining flights; estimated onset compared with the onset logged inside each flight.
- Implementation lives in `src/sih_model.py`; this notebook runs the experiments through it and presents the committed result files.

Result files (under `reports/v4`): `e2_leave_one_subtype_out.csv`, `e2_per_seed.csv`, `e2_splits.csv`, `e2_held_*_flight_scores.csv`, `e2_seen_*_flight_scores.csv`, `e2_gate_cal_scores_*.csv`, `e5_onset_localisation.csv`, `e5_onset_summary.csv`. Manuscript: Sections 6.4 and 6.5, Tables 6, 6b, 6c, 6d and 7d, and the hold-out, gate and onset figures.

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
RUN, FEAT, FORCE = "v4", "features_v3", False
subprocess.run(["pip", "install", "-q", "xgboost", "scikit-learn", "scipy", "pandas"], check=True)
out = P.REPORTS / RUN
import pandas as pd
def e2_complete():
    f = out / "e2_leave_one_subtype_out.csv"
    return f.exists() and (out / "e2_per_seed.csv").exists() and "unseen_gate_gaprule_withheld_mean" in pd.read_csv(f).columns
def e5_complete():
    f = out / "e5_onset_summary.csv"
    return f.exists() and "method" in pd.read_csv(f).columns
todo = [e for e, ok in (("e2", e2_complete()), ("e5", e5_complete())) if FORCE or not ok]
print("running:", todo or "nothing (outputs exist; set FORCE = True to recompute)")
if todo:
    proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_model.py"), "--features", str(P.FEATURES / FEAT), "--out", str(out),
                             "--only", ",".join(todo), "--e2_reps", "3"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    print("exit code:", proc.wait())


In [ ]:
# ---- E2: same fitted pipeline on the withheld subtype and on the disjoint seen-subtype test set ----
import pandas as pd, numpy as np
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
SUB = {"jump": "Jump", "drift_incoherent": "Incoherent drift", "drift_coherent": "Coherent drift", "off_then_ok": "Receiver silence",
       "stuck_then_ok": "Frozen receiver output", "no_fix_then_ok": "No-fix reporting", "baro_stuck": "Frozen barometer", "mag_stuck": "Frozen magnetometer"}
lo = pd.read_csv(out / "e2_leave_one_subtype_out.csv")
ps = pd.read_csv(out / "e2_per_seed.csv")
pm = lambda a, b: [f"{m:.3f} ± {s:.3f}" for m, s in zip(lo[a], lo[b])]
comparison = pd.DataFrame({
    "held-out subtype": lo["held_out_subtype"].map(SUB),
    "n unseen / n seen-test": [f"{int(a)} / {int(b)}" for a, b in zip(lo["unseen_n_mean"], lo["seen_n_mean"])],
    "fine accuracy unseen": pm("unseen_acc_mean", "unseen_acc_std"), "fine accuracy seen": pm("seen_acc_mean", "seen_acc_std"),
    "coarse accuracy unseen": pm("unseen_coarse_acc_mean", "unseen_coarse_acc_std"), "coarse accuracy seen": pm("seen_coarse_acc_mean", "seen_coarse_acc_std"),
    "mean confidence unseen": pm("unseen_mean_conf_mean", "unseen_mean_conf_std"), "mean confidence seen": pm("seen_mean_conf_mean", "seen_mean_conf_std"),
    "fine coverage 0.10 unseen": pm("unseen_coverage_a0.1_mean", "unseen_coverage_a0.1_std"), "fine coverage 0.10 seen": pm("seen_coverage_a0.1_mean", "seen_coverage_a0.1_std"),
})
print(f"seeds: {ps.seed.nunique()}")
display(comparison)
splits = pd.read_csv(out / "e2_splits.csv")
display(splits[splits.seed == 0].drop(columns=["seed"]).pivot_table(index="held_out_subtype", columns="family", values=["n_train", "n_cal", "n_seen_test"], aggfunc="first"))


In [ ]:
# ---- E4: distributional abstention gate ----
gate = pd.DataFrame({
    "held-out subtype": lo["held_out_subtype"].map(SUB),
    "conformal policy alone, unseen": pm("unseen_operational_abstain_a0.1_mean", "unseen_operational_abstain_a0.1_std"),
    "Mahalanobis gate, unseen": pm("unseen_gate_mahal_withheld_mean", "unseen_gate_mahal_withheld_std"),
    "Mahalanobis gate, seen": pm("seen_gate_mahal_withheld_mean", "seen_gate_mahal_withheld_std"),
    "isolation-forest gate, unseen": pm("unseen_gate_iso_withheld_mean", "unseen_gate_iso_withheld_std"),
    "isolation-forest gate, seen": pm("seen_gate_iso_withheld_mean", "seen_gate_iso_withheld_std"),
    "conformal or Mahalanobis, unseen": pm("unseen_gate_mahal_or_conformal_withheld_mean", "unseen_gate_mahal_or_conformal_withheld_std"),
    "conformal or Mahalanobis, seen": pm("seen_gate_mahal_or_conformal_withheld_mean", "seen_gate_mahal_or_conformal_withheld_std"),
    **({"gap-rule baseline, unseen": pm("unseen_gate_gaprule_withheld_mean", "unseen_gate_gaprule_withheld_std"),
        "gap-rule baseline, seen": pm("seen_gate_gaprule_withheld_mean", "seen_gate_gaprule_withheld_std")}
       if "unseen_gate_gaprule_withheld_mean" in lo.columns else {}),
})
display(gate)
# threshold-free separability: from this run if present, otherwise from the E4b replay table (05_gate_separability.ipynb)
if "unseen_gate_mahal_auroc_mean" in lo.columns:
    separability = pd.DataFrame({
        "held-out subtype": lo["held_out_subtype"].map(SUB),
        "Mahalanobis AUROC": pm("unseen_gate_mahal_auroc_mean", "unseen_gate_mahal_auroc_std"),
        "Mahalanobis catch at 5% FPR": pm("unseen_gate_mahal_catch_at_5pct_fpr_mean", "unseen_gate_mahal_catch_at_5pct_fpr_std"),
        "isolation-forest AUROC": pm("unseen_gate_iso_auroc_mean", "unseen_gate_iso_auroc_std"),
        "plain confidence AUROC": pm("unseen_gate_one_minus_conf_auroc_mean", "unseen_gate_one_minus_conf_auroc_std"),
        "plain confidence catch at 5% FPR": pm("unseen_gate_one_minus_conf_catch_at_5pct_fpr_mean", "unseen_gate_one_minus_conf_catch_at_5pct_fpr_std"),
    })
    display(separability)
elif (out / "tables" / "t6e_gate_separability.csv").exists():
    sep = pd.read_csv(out / "tables" / "t6e_gate_separability.csv")
    keep = [c for c in ["held_out_subtype", "score", "auroc_mean", "auroc_lo_mean", "auroc_hi_mean", "tpr_at_fpr_mean", "withheld_unseen_q95_mean", "withheld_seen_q95_mean"] if c in sep.columns]
    display(sep[keep].round(3))
else:
    print("separability table not available yet")


In [ ]:
# ---- E5: onset localisation against the logged onset ----
summ = pd.read_csv(out / "e5_onset_summary.csv")
summ["subtype"] = summ["subtype"].map(lambda v: SUB.get(v, v))
display(summ.round(3))
det = pd.read_csv(out / "e5_onset_localisation.csv")
if "method" in det.columns:
    det = det[det.method == "rule"]
att = det[(det.family != "nominal") & det.onset_error_s.notna()]
display(att.groupby("subtype")["onset_error_s"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1))
nom = det[det.family == "nominal"]
print("nominal flights:", len(nom), "| with at least one alert episode:", int(nom["false_alert"].sum()), "| mean episodes per nominal flight:", round(nom["n_alert_episodes"].mean(), 2))
